# Interest Rate Swaps

This notebook covers vanilla IRS pricing and curve bootstrapping using `bond.swaps`.

**Topics**
- Bootstrapping a discount curve from deposits and swap rates
- Zero rates and forward rates from the bootstrapped curve
- Pricing a fixed-for-floating swap
- Par rates and annuity factors
- DV01 for swap books

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/bond/blob/main/notebooks/03_swaps.ipynb)

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
from bond import bootstrap, price_irs, dv01_irs, par_rate, annuity, forward_rate

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('bond.swaps loaded ✓')

## Market Data

In [ ]:
# Short end: deposit rates (simple interest, Act/360)
deposits = {
    0.25: 0.0530,   # 3M
    0.5:  0.0525,   # 6M
}

# Swap par rates (semi-annual fixed vs SOFR floating)
swaps = {
    1:  0.0510,
    2:  0.0490,
    3:  0.0475,
    5:  0.0455,
    7:  0.0445,
    10: 0.0440,
    15: 0.0435,
    20: 0.0432,
    30: 0.0430,
}

# Bootstrap the curve
curve = bootstrap(deposits, swaps)
print(f'Bootstrapped: {curve}')

## Bootstrapping

Bootstrapping extracts zero-coupon discount factors from market instruments sequentially:

1. **Short end** (deposits): $P(0,t) = \frac{1}{1 + r \cdot t}$
2. **Swap instruments**: for each par swap with rate $K$ and maturity $T$, solve:
   $$P(0,T) = \frac{1 - K \cdot \Delta \sum_{i=1}^{T-1} P(0,t_i)}{1 + K \cdot \Delta}$$

In [ ]:
t = np.linspace(0.1, 30, 300)
zero_rates = curve.zero_rate(t) * 100
disc_factors = curve.df(t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(t, zero_rates, lw=2, color='#1565C0', label='Zero rates')
# Overlay par rates
par_mats = list(swaps.keys())
par_rates_vals = [par_rate(curve, float(m))*100 for m in par_mats]
ax1.scatter(par_mats, list(np.array(list(swaps.values()))*100),
            color='red', zorder=5, s=40, label='Market swap rates')
ax1.scatter(par_mats, par_rates_vals, color='green', zorder=5, marker='x',
            s=60, label='Bootstrapped par rates')
ax1.set_xlabel('Maturity (years)'); ax1.set_ylabel('Rate (%)')
ax1.set_title('Zero Curve vs Par Swap Rates')
ax1.legend(fontsize=8)

ax2.plot(t, disc_factors, lw=2, color='#2E7D32')
ax2.set_xlabel('Maturity (years)'); ax2.set_ylabel('Discount Factor')
ax2.set_title('Discount Curve P(0,t)')
plt.tight_layout()
plt.show()

print('\nBootstrapped par rates vs market (should match):')
for T, K_mkt in swaps.items():
    K_boot = par_rate(curve, float(T))
    print(f'  {T:2}y: market={K_mkt:.4%}  bootstrapped={K_boot:.4%}  diff={abs(K_boot-K_mkt)*1e4:.2f}bp')

## Forward Rates

In [ ]:
fwd_rates = np.array([curve.forward_rate(max(0.01, ti-0.25), ti+0.25) * 100 for ti in t])

fig, ax = plt.subplots()
ax.plot(t, zero_rates, lw=2, label='Zero curve')
ax.plot(t, fwd_rates, lw=2, ls='--', label='Forward curve')
ax.set_xlabel('Maturity (years)'); ax.set_ylabel('Rate (%)')
ax.set_title('Zero and Forward Rates from Bootstrapped Curve')
ax.legend()
plt.tight_layout()
plt.show()

print('Selected 1Y forward rates:')
for t1, t2 in [(0,1),(1,2),(2,3),(3,5),(5,7),(7,10)]:
    f = forward_rate(curve, max(t1, 0.01), t2)
    print(f'  f({t1}y, {t2}y) = {f:.4%}')

## Pricing a Vanilla IRS

A fixed-for-floating swap (payer perspective):

$$NPV = N \cdot \left[(1 - P(0,T)) - K \cdot A(0,T)\right]$$

where $A(0,T) = \Delta \sum_i P(0,t_i)$ is the annuity factor.

In [ ]:
notional = 10_000_000  # $10M
maturity = 5.0
K_par = par_rate(curve, maturity)
A = annuity(curve, maturity)

print(f'5Y par swap rate:  {K_par:.4%}')
print(f'5Y annuity factor: {A:.6f}')
print()

# Price swaps at various fixed rates
fixed_rates = np.linspace(0.03, 0.07, 100)
npvs = [price_irs(curve, notional, K, maturity, pay_fixed=True) for K in fixed_rates]

fig, ax = plt.subplots()
ax.plot(fixed_rates * 100, np.array(npvs) / 1e6, lw=2, color='#1565C0')
ax.axhline(0, color='black', lw=0.8)
ax.axvline(K_par * 100, color='red', ls='--', lw=1.5, label=f'Par rate = {K_par:.4%}')
ax.set_xlabel('Fixed Rate (%)')
ax.set_ylabel('NPV ($M)')
ax.set_title(f'5Y Payer IRS NPV vs Fixed Rate — $10M Notional')
ax.legend()
plt.tight_layout()
plt.show()

# Specific scenarios
print('NPV scenarios (5Y payer, $10M notional):')
for K in [0.040, K_par, 0.050, 0.060]:
    npv = price_irs(curve, notional, K, maturity, pay_fixed=True)
    label = '← par rate' if abs(K - K_par) < 1e-4 else ''
    print(f'  Fixed={K:.4%}  NPV=${npv:>12,.0f}  {label}')

## Swap DV01

In [ ]:
maturities = [1, 2, 3, 5, 7, 10, 15, 20, 30]
dv01s = [dv01_irs(curve, 1_000_000, par_rate(curve, float(m)), float(m)) for m in maturities]

fig, ax = plt.subplots()
ax.bar(maturities, dv01s, color='#1565C0', alpha=0.8, width=0.8)
ax.set_xlabel('Swap Maturity (years)')
ax.set_ylabel('DV01 ($)')
ax.set_title('IRS DV01 by Maturity — $1M Notional, struck at-par')
for m, d in zip(maturities, dv01s):
    ax.text(m, d + 1, f'${d:.0f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('DV01 table ($1M notional at-par swaps):')
print(f'  {"Tenor":>6}  {"Par rate":>9}  {"DV01 ($)":>10}')
for m, d in zip(maturities, dv01s):
    K = par_rate(curve, float(m))
    print(f'  {m:>5}y  {K:>9.4%}  {d:>10.2f}')